In [1]:
import torch
from torch import nn
import numpy as np
import gym
import random
from copy import deepcopy
import collections

setattr(np, "bool8", np.bool)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
class Model(nn.Module):
    def __init__(self, obs_n=4, act_n=2):
        super().__init__()
        self.net = torch.nn.Sequential(
            nn.Linear(obs_n, 128),
            nn.ReLU(),
            nn.Linear(128, act_n)
        )
    
    def forward(self, obs):
        return self.net(obs)


In [3]:
class ReplayBuffer:
    def __init__(self, max_len):
        self.buffer = collections.deque(maxlen=max_len)

    def append(self, item):
        self.buffer.append(item)

    def sample(self, minibatch):
        exps = random.sample(self.buffer, minibatch)
        # sarsa
        obs_list, action_list, reward_list, next_obs_list, done_list = (
            [],
            [],
            [],
            [],
            [],
        )
        for exp in exps:
            obs, action, reward, next_obs, done = exp
            obs_list.append(obs)
            action_list.append(action)
            reward_list.append(reward)
            next_obs_list.append(next_obs)
            done_list.append(done)
        
        return np.array(obs_list), np.array(action_list), np.array(reward_list), np.array(next_obs_list), np.array(done_list)

    def __len__(self):
        return len(self.buffer)

In [4]:
class DQN:
    def __init__(self, model, replay_buffer, gamma=0.9, lr=0.3):
        self.model:nn.Module = model
        self.target_model:nn.Module = deepcopy(self.model)
        self.replay_buffer = replay_buffer
        self.gamma = gamma
        self.optimizer = torch.optim.Adam(lr=lr, params=self.model.parameters())
        self.loss_fn = nn.MSELoss()

    def learn(self, obs, action, reward, next_obs, done):
        obs = torch.tensor(obs, dtype=torch.float)
        action = torch.tensor(action, dtype=torch.long)
        reward = torch.tensor(reward, dtype=torch.float)
        next_obs = torch.tensor(next_obs, dtype=torch.float)
        done = torch.tensor(done, dtype=torch.float)

        with torch.no_grad():
            target_score = self.target_model(next_obs)

            target_score = torch.max(target_score, dim=1)[0]
            target_score = reward + (1 - done) * self.gamma * target_score
        
        pred_q = self.model(obs)
        batch = pred_q.shape[0]
        batch_index = torch.arange(batch)

        pred_q = pred_q[batch_index, action]

        loss = self.loss_fn(pred_q, target_score)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss
    
    def sync_parameter(self):
        self.target_model.load_state_dict(self.model.state_dict())


In [5]:
class Agent:
    def __init__(self, alg:DQN, act_n=2, update_target_steps=200):
        self.alg = alg
        self.epsilon = 0.1
        self.act_n = act_n
        self.update_target_steps = update_target_steps
        self.total_step = 0

    
    def sample(self, obs):
        if np.random.rand() < self.epsilon:
            return np.random.choice(self.act_n)

        return self.predict(obs)
    
    def predict(self, obs):

        obs = torch.tensor(obs).unsqueeze(0)
        self.alg.model.eval()
        with torch.no_grad():
            pred = self.alg.model(obs)

            pred = torch.argmax(pred, dim=1)

        self.alg.model.train()
        return int(pred.item())
    
    def learn(self, obs, action, reward, next_obs, done):
        if self.total_step % self.update_target_steps == 0:
            self.alg.sync_parameter()
        
        return self.alg.learn(obs, action, reward, next_obs, done)

In [6]:
def run_one_episode(agent, env, replay_buffer):
    obs = env.reset()[0]
    MEMORY_WARMUP_SIZE = 200
    mini_batch = 100
    total_loss = 0
    total_reward = 0
    while True:

        action = agent.sample(obs)
        next_observation, reward, done, _, _ = env.step(action)
        replay_buffer.append((obs, action, reward, next_observation, done))

        agent.total_step += 1
        
        if len(replay_buffer) > MEMORY_WARMUP_SIZE:
            obs_list, action_list, reward_list, next_obs_list, done_list = replay_buffer.sample(mini_batch)
            loss = agent.learn(obs_list, action_list, reward_list, next_obs_list, done_list)

            total_loss += loss
        total_reward += reward
        
        if done:
            break
        else:
            obs = next_observation
    return total_loss, total_reward

In [7]:
env = gym.make("CartPole-v0")
replay_buffer = ReplayBuffer(max_len=1000)
alg = DQN(Model(), replay_buffer, lr=1e-3)
agent = Agent(alg)

for i in range(5000):
    total_loss, total_reward = run_one_episode(agent, env, replay_buffer)
    print(f"Episode:{i+1}, Total Loss:{total_loss}, Total Reward:{total_reward}")

d:\software\miniconda3\Lib\site-packages\gym\envs\registration.py:555: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(


Episode:1, Total Loss:0, Total Reward:23.0
Episode:2, Total Loss:0, Total Reward:29.0
Episode:3, Total Loss:0, Total Reward:30.0
Episode:4, Total Loss:0, Total Reward:20.0
Episode:5, Total Loss:0, Total Reward:16.0
Episode:6, Total Loss:0, Total Reward:19.0
Episode:7, Total Loss:0, Total Reward:27.0
Episode:8, Total Loss:0, Total Reward:24.0
Episode:9, Total Loss:12.1679048538208, Total Reward:29.0
Episode:10, Total Loss:4.833706378936768, Total Reward:18.0
Episode:11, Total Loss:0.983255922794342, Total Reward:21.0
Episode:12, Total Loss:0.2143675684928894, Total Reward:17.0
Episode:13, Total Loss:0.18511849641799927, Total Reward:16.0
Episode:14, Total Loss:0.167301207780838, Total Reward:16.0
Episode:15, Total Loss:0.12985701858997345, Total Reward:13.0
Episode:16, Total Loss:0.10563386231660843, Total Reward:11.0
Episode:17, Total Loss:0.0785331130027771, Total Reward:8.0
Episode:18, Total Loss:0.10142053663730621, Total Reward:9.0
Episode:19, Total Loss:0.18406710028648376, Total 